# Context-I2W 源码精读：组合图像检索是怎么实现的

这个 notebook 里**所有代码单元都是官方源码逐字截取**，没有改写、没有简化、没有删行。
每格的注释头标了出处文件与行号。

- 仓库：`Pter61/context-i2w`
- 提交：`c524d5b`

**讲什么**：模型本体、IVS、VTE、伪词如何插回文本、训练目标、推理路径。
**不讲什么**：数据加载（`src/data.py`）、训练调度、日志、分布式、命令行参数。

---

## 一句话看懂这个方法

普通组合检索把图像特征和文本特征在**向量层**融合：

```text
image_feat (768) ──┐
                   ├──> 拼接 / 相加 ──> 检索向量
text_feat  (768) ──┘
```

Context-I2W 把融合提前到**词这一层**：

```text
image_tokens (257×1024) ──┐
                          ├──> 生成一个"伪词" (768) ──> 替换句子里的 <|replace|> ──> 文本编码器 ──> 检索向量
text_cls     (768)      ──┘
```

**为什么这么设计**：生成的伪词活在 CLIP 的词嵌入空间里，经过文本编码器后得到的向量，
天然和 CLIP 的图像特征是同一个空间的。不需要再学一个融合头，也不会出现"融合后的向量
不在 CLIP 空间里"的问题。

论文管这个"按语境生成的词"叫 **context-dependent word**，管生成它的模块叫
**VTE**（Visual-to-Text Encoder）。管"从图像 token 里挑出与修改意图相关的信息"的模块叫
**IVS**（Image Visual Selector）。

## 1. 整体架构

```text
┌──────────────────── CLIP ViT-L/14（冻结，427.6 M 参数）────────────────────┐
│                                                                            │
│  图像 (3,224,224)                                                          │
│      │                                                                     │
│      ▼  conv1 14×14 stride14 ──> 16×16=256 个 patch                        │
│   [CLS] + 256 patch = 257 个 token，每个 1024 维                            │
│      │                                                                     │
│      ├── 过 24 层 Transformer ──> encode_image       → 取 CLS → 768 维      │
│      └── 过 24 层 Transformer ──> encode_image_tokens → 257×1024（不降维）  │
│                                                                            │
│  文本 (1,77) → 词嵌入 768 → +位置编码 → 12 层 Transformer → 取 EOT         │
│      │                                                                     │
│      └──> text_projection (768→768) ──> encode_text  → 768 维              │
└────────────────────────────────────────────────────────────────────────────┘
                                    │
                     图像 token (B,257,1024) + 文本 CLS (B,768)
                                    │
                                    ▼
┌──────────── IM2TEXT_OWM_Attn（可训练，68.5 M 参数）─────────────┐
│                                                                │
│  to_c: 文本 768 → 1024，升到视觉空间                            │
│  W_R : 1024×1024 线性 + weight_norm      ← 这就是 IVS           │
│        x = W_R(x)       每个视觉 token 各自变换                │
│        x = x * c        按文本的 1024 个分量逐通道加权          │
│                                                                │
│  ┌── 全局分支 ──────────┐   ┌── 局部分支 ───────────────────┐  │
│  │ 取 CLS token          │   │ 取 256 个 patch token          │  │
│  │ ln_post → proj        │   │ 4 个可学习 latent × 6 层      │  │
│  │ MLP(768→512→512→768)  │   │ cross-attention → 4×1024      │  │
│  └───────────┬───────────┘   │ 平均池化 → 1024               │  │
│              │               │ ln_post_local → proj_local    │  │
│              │               │ MLP → 768                     │  │
│              │               └───────────┬───────────────────┘  │
│              │                           │ × tanh(local_gate)   │
│              └─────────── + ─────────────┘                      │
│                         ▼                                       │
│                    伪词向量 (B,768)                              │
└─────────────────────────────────────────────────────────────────┘
                         │
                         ▼
   替换句子 "a photo of <|replace|> , cap2 and cap1" 里的那个特殊 token
                         │
                         ▼
              再过文本 Transformer → 取 EOT → 组合查询向量

### 关键事实

| 模块 | 参数量 | 训练时是否更新 |
|---|---:|---|
| CLIP ViT-L/14（`state_dict`） | 427,616,513 | **否**，全程 `torch.no_grad()` |
| IM2TEXT_OWM_Attn（`state_dict_img2text`） | 68,471,297 | **是**，优化器里只有它 |

（参数量是从 `epoch_30.pt` 里实际数出来的，不是估的。）

所以这个方法的"训练"其实只训练了一个 68 M 的适配器，CLIP 完全不动。

**训练数据是 Conceptual Captions（CC3M）**，官方 README 的原话是
"We utilize Conceptual Captions URLs to train a model"。`data/README.md` 里的
COCO 和 ImageNet-R 是**零样本测试集**，不是训练集。FashionIQ 同样只用于评估——
这就是"零样本"的实际含义：核心检索模型从没见过 FashionIQ 的图片和标注。

## 2. 训练路径与推理路径不一样

这一点很容易看错，先摆出来：

| | 训练 | 推理 |
|---|---|---|
| 视觉输入 | 参考图 | 参考图 |
| 文本输入 | 完整句子 `a photo of <|replace|> , cap2 and cap1` | 同一个模板 |
| 伪词放到哪 | **插入**到 `a photo of` 的末尾 | **替换**模板里的 `<|replace|>` |
| 对比目标 | 参考图的 CLIP 图像特征 | —— |
| 损失 | 对称交叉熵（CLIP 式对比损失） | —— |

训练做的事：让"`a photo of <伪词>`"这句话的文本嵌入，尽可能接近参考图的图像特征。
推理做的事：把这个伪词塞回**带修改描述的完整句子**里，再去检索。

也就是说，伪词先在"描述这张图"的任务上被训练成一个 CLIP 能读懂的东西，
然后才被拿去当句子里的成分。这个不对称是论文的设计，不是笔误。

下面按数据流顺序逐个模块读。

## 3. 视觉端：为什么要 `get_tokens`

`VisualTransformer` 类里有**两套出口**：

- `forward` —— 返回**只有 CLS** 的 768 维向量（经过 `ln_post` 和 `proj` 降维）→ 用于算相似度
- `get_tokens` —— 返回**全部 257 个 token** 的 1024 维（**不**经过 `ln_post` / `proj`）→ 用于喂给 VTE

为什么不能只用 CLS？因为全局池化已经把空间信息压碎了。修改描述说的往往是
"把袖子改短""换成红条纹"这种**局部**变化，VTE 需要看到 16×16 个 patch 各自的表示，
才能判断"改的是哪一块"。所以必须用 `get_tokens`。

两个函数的前半段完全一样，只有最后几行不同——**这正是要并排看的重点**。

In [ ]:
# 出处：model/model.py  第 491-524 行（官方源码逐字截取）
    def forward(self, x: torch.Tensor):
        x = self.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat(
            [self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device),
             x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)

        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD

        x = self.ln_post(x[:, 0, :])

        if self.proj is not None:
            x = x @ self.proj

        return x

    def get_tokens(self, x: torch.Tensor):
        x = self.conv1(x)  # shape = [*, width, grid, grid]
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat(
            [self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device),
             x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        return x

### 读法

上面这 30 行里，前 20 行是完全重复的（patch 切分 → 加 CLS → 加位置编码 → 24 层 Transformer）。

**分叉点在最后两行**：

```python
# forward：只取第 0 个 token（CLS），再降维到 768
x = self.ln_post(x[:, 0, :])
if self.proj is not None:
    x = x @ self.proj
return x

# get_tokens：直接返回全部 257 个 token，1024 维，不降维
return x
```

几个形状要记住：

| 量 | 值 | 来源 |
|---|---|---|
| 输入分辨率 | 224 | `input_resolution` |
| patch 大小 | 14 | `patch_size` |
| 网格 | 224/14 = 16 | `grid` |
| patch 数 | 16×16 = 256 | |
| 加 CLS 后 | **257** | `[CLS] + 256` |
| 视觉宽度 | **1024** | `width`（ViT-L/14） |
| 输出维度 | **768** | `output_dim` = `embed_dim` |

> 注意：ViT-L/14 的视觉宽度是 **1024**，文本宽度是 **768**。这两个数不相等，
> 所以后面到处需要投影层。这是读这段代码时最常见的困惑来源。

## 4. 文本端：`encode_text` 怎么取句子向量

三个入口一起看：

- `encode_image` —— 调 `VisualTransformer.forward`
- `encode_image_tokens` —— 调 `VisualTransformer.get_tokens`
- `encode_text` —— 词嵌入 → 文本 Transformer → **取 EOT 位置** → `text_projection`

取 EOT（End Of Text）是 CLIP 的标准做法：序列里最后一个有意义的 token 位置上，
Transformer 已经"看过"整句。代码用 `text == self.end_id` 定位它，
而不是写死第 77 个位置——因为序列是补零到 77 的，EOT 的位置随句子长度变化。

In [ ]:
# 出处：model/model.py  第 643-662 行（官方源码逐字截取）
    def encode_image(self, image):
        return self.visual(image.type(self.dtype))

    def encode_image_tokens(self, image):
        return self.visual.get_tokens(image.type(self.dtype))

    def encode_text(self, text):
        x = self.token_embedding(text).type(self.dtype)  # [batch_size, n_ctx, d_model]

        x = x + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)
        # x.shape = [batch_size, n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        collect_ind = text == self.end_id
        collect_ind = collect_ind.nonzero()[:, 1]
        x = x[torch.arange(x.size(0)), collect_ind] @ self.text_projection
        return x

`collect_ind = (text == self.end_id).nonzero()[:, 1]` 这一行值得单独看：

- `text` 形状 `(B, 77)`，每条文本恰好有一个 EOT
- `nonzero()` 返回 `(B, 1)` 的坐标，`[:, 1]` 取列号，就是 EOT 在 77 个位置里的第几个
- `x[torch.arange(x.size(0)), collect_ind]` 按批取对应位置的向量

`self.end_id = vocab_size - 1`，`self.replace_id = vocab_size - 3`——
这两个 ID 是 CLIP 词表里人为留出来的特殊 token，后面会用到 `replace_id`。

## 5. IVS：`W_R` 是什么

这是论文的第一个关键设计。构造函数里只有两行，但含义很重：

```python
self.W_R = nn.Linear(vision_width, vision_width, bias=False)  # Selector
self.W_R = torch.nn.utils.weight_norm(self.W_R)               # normalization
```

- `vision_width = 1024`，所以这是一个 **1024×1024** 的方阵
- `bias=False`：纯线性变换，不做平移
- `weight_norm`：把权重拆成"方向"和"长度"两部分分别学习。这是 2016 年的老技巧，
  官方代码原样保留，新版 PyTorch 会打 deprecation 警告，**不影响数值**

源码注释里它叫 `Selector`，论文里叫 **IVS（Image Visual Selector）**，
`forward` 里那行注释叫 **Context Rotator**——三个名字指同一个东西。

In [ ]:
# 出处：model/model.py  第 198-245 行（官方源码逐字截取）
class IM2TEXT_OWM_Attn(nn.Module):
    def __init__(self,
                 transformer_width: int,
                 vision_width: int,
                 embed_dim=512, middle_dim=512, output_dim=512, n_layer=2, dropout=0.1, stride=1):
        super().__init__()

        self.to_c = nn.Linear(transformer_width, vision_width, bias=False)
        self.W_R = nn.Linear(vision_width, vision_width, bias=False)  # Selector
        self.W_R = torch.nn.utils.weight_norm(self.W_R)  # normalization

        scale = vision_width ** -0.5
        self.ln_post = LayerNorm(vision_width)
        self.ln_post_local = LayerNorm(vision_width)
        self.proj = nn.Parameter(scale * torch.randn(vision_width, output_dim))
        self.proj_local = nn.Parameter(scale * torch.randn(vision_width, output_dim))

        # local information
        self.VT_Extractor = VisualTargetExtractor(dim=vision_width)

        # global
        self.fc_out = nn.Linear(middle_dim, output_dim)
        layers = []
        dim = embed_dim
        for _ in range(n_layer):
            block = []
            block.append(nn.Linear(dim, middle_dim))
            block.append(nn.Dropout(dropout))
            block.append(nn.ReLU())
            dim = middle_dim
            layers.append(nn.Sequential(*block))
        self.layers = nn.Sequential(*layers)

        # local
        self.fc_out_local = nn.Linear(middle_dim, output_dim)
        layers_local = []
        dim = embed_dim
        for _ in range(n_layer):
            block_local = []
            block_local.append(nn.Linear(dim, middle_dim))
            block_local.append(nn.Dropout(dropout))
            block_local.append(nn.ReLU())
            dim = middle_dim
            layers_local.append(nn.Sequential(*block_local))
        self.layers_local = nn.Sequential(*layers_local)

        self.avgpool = nn.AvgPool2d(kernel_size=(4, 1))
        self.local_gate = nn.Parameter(torch.tensor([0.0]))

### 初始化里都有什么

| 成员 | 形状 | 作用 |
|---|---|---|
| `to_c` | 768 → 1024 | 把文本 CLS 升到视觉空间 |
| **`W_R`** | **1024 → 1024** | **IVS：按语境重构视觉 token** |
| `ln_post` / `proj` | 1024 / 1024×768 | 全局分支的归一化与降维 |
| `ln_post_local` / `proj_local` | 同上 | 局部分支的归一化与降维 |
| `VT_Extractor` | 见第 6 节 | 局部分支的注意力提取器 |
| `layers` / `layers_local` | 768→512→512 | 两个分支各一套 MLP |
| `fc_out` / `fc_out_local` | 512→768 | 两个分支各自的输出层 |
| `avgpool` | kernel=(4,1) | 把 4 个 latent 池成 1 个 |
| **`local_gate`** | **标量，初值 0** | **局部支路的门控** |

`local_gate` 初值是 **0**，`tanh(0) = 0`。也就是说训练开始时局部分支的贡献是 **0**，
网络先只用全局特征，再慢慢学会"局部信息值多少"。这是一个很稳的初始化技巧。

## 6. 局部分支：4 个 latent 从 256 个 patch 里"读出"摘要

局部分支要解决的问题是：**256 个 patch 太多了，直接展平会淹没关键信息**。
论文的做法是借鉴 Perceiver / Flamingo 的 resampler：
用 **4 个可学习的 query**，通过 **6 层 cross-attention**，从 256 个 patch 里提取 4 个摘要向量。

下面三个类是一套：`FeedForward`（前馈块）、`ContextDependantAttention`（注意力的核心）、
`VisualTargetExtractor`（把两者叠 6 层）。

In [ ]:
# 出处：model/model.py  第 99-106 行（官方源码逐字截取）
def FeedForward(dim, mult=4):
    inner_dim = int(dim * mult)
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, inner_dim, bias=False),
        nn.GELU(),
        nn.Linear(inner_dim, dim, bias=False),
    )

In [ ]:
# 出处：model/model.py  第 109-150 行（官方源码逐字截取）
class ContextDependantAttention(nn.Module):
    def __init__(self, *, dim, dim_head=64, heads=8):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        inner_dim = dim_head * heads

        self.norm_media = nn.LayerNorm(dim)
        self.norm_latents = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(dim, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

    def forward(self, x, latents):
        """
        Args:
            x (torch.Tensor): image features
                shape (b, T, n1, D)
            latent (torch.Tensor): latent features
                shape (b, T, n2, D)
        """
        x = self.norm_media(x)
        latents = self.norm_latents(latents)

        h = self.heads

        q = self.to_q(latents)
        kv_input = torch.cat((x, latents), dim=-2)
        k, v = self.to_kv(kv_input).chunk(2, dim=-1)
        q, k, v = rearrange_many((q, k, v), "b t n (h d) -> b h t n d", h=h)
        q = q * self.scale

        # attention
        sim = einsum("... i d, ... j d  -> ... i j", q, k)
        sim = sim - sim.amax(dim=-1, keepdim=True).detach()
        attn = sim.softmax(dim=-1)
        # print(attn[0][0][0][1], attn.shape)
        out = einsum("... i j, ... j d -> ... i d", attn, v)
        out = rearrange(out, "b h t n d -> b t n (h d)", h=h)
        return self.to_out(out)

### `ContextDependantAttention` 的关键一行

```python
kv_input = torch.cat((x, latents), dim=-2)
```

query 只来自 latents，key/value 来自 **图像 token 和 latents 拼接**。
这就是类名里 "Context Dependant" 的含义：latents 既看图像，也看自己上一轮的结论。

后面的 `sim - sim.amax(...).detach()` 是标准的数值稳定性写法（等价于 softmax 前的平移），
`detach()` 保证梯度不回传到最大值那一项——这是 ViT 里的常见技巧。

In [ ]:
# 出处：model/model.py  第 152-195 行（官方源码逐字截取）
class VisualTargetExtractor(nn.Module):
    def __init__(
            self,
            *,
            dim,
            depth=6,
            dim_head=64,
            heads=8,
            num_latents=4,
            ff_mult=4,
    ):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latents, dim))

        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(
                nn.ModuleList(
                    [
                        ContextDependantAttention(dim=dim, dim_head=dim_head, heads=heads),
                        FeedForward(dim=dim, mult=ff_mult),
                    ]
                )
            )

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): image features
                shape (b, T, F, v, D)
        Returns:
            shape (b, T, n, D) where n is self.num_latents
        """
        b, T, F, v = x.shape[:4]

        # blocks
        latents = repeat(self.latents, "n d -> b T n d", b=b, T=T)
        for attn, ff in self.layers:
            latents = attn(x, latents) + latents
            latents = ff(latents) + latents

        return self.norm(latents)

### `VisualTargetExtractor`

- `self.latents`：4 个 1024 维的可学习向量，是所有查询共享的"探针"
- `repeat(..., "n d -> b T n d")`：复制到 batch 和序列维度
- 每层 `attn(x, latents) + latents` → `ff(latents) + latents`，是标准的 pre-norm 残差块
- 输出 `(b, T, 4, 1024)`

为什么是 4 个而不是 1 个？多个 latent 可以分别关注不同区域（比如一边关注领口、
一边关注袖子），最后再池化合并，比单个 latent 更有表达力。

## 7. VTE 总装：两条支路 + 一个门控

现在把前面所有零件拼起来。下面是 `IM2TEXT_OWM_Attn.forward`，只有 30 行多一点，但每一步都不能跳过。

In [ ]:
# 出处：model/model.py  第 247-279 行（官方源码逐字截取）
    def forward(self, c: torch.Tensor, x: torch.Tensor):
        c = self.to_c(c)
        c = c.unsqueeze(1)  # LD -> LND

        # Context Rotator
        x = self.W_R(x)
        # element-wise multiplication
        x = torch.mul(x, c)

        x_local = self.VT_Extractor(x[:, 1:, :].unsqueeze(1))
        # fusion
        x_local = self.avgpool(x_local)
        x_local = x_local.squeeze(1).squeeze(1)

        # get global image embedding
        x_global = self.ln_post(x[:, 0, :])

        if self.proj is not None:
            x_global = x_global @ self.proj

        for layer in self.layers:
            x_global = layer(x_global)
        x_global = self.fc_out(x_global)

        # get local image embeddings
        x_local = self.ln_post_local(x_local)
        if self.proj_local is not None:
            x_local = x_local @ self.proj_local
        for layer in self.layers_local:
            x_local = layer(x_local)
        x_local = self.fc_out_local(x_local)
        x = x_global + (x_local * self.local_gate.tanh())
        return x

### 逐步跟踪形状

输入：`c` 是文本 CLS `(B, 768)`，`x` 是视觉 token `(B, 257, 1024)`。

| 行 | 操作 | 结果形状 |
|---|---|---|
| `c = self.to_c(c)` | 768 → 1024 | `(B, 1024)` |
| `c = c.unsqueeze(1)` | 加一个 token 维 | `(B, 1, 1024)` |
| `x = self.W_R(x)` | 每个 token 各自做 1024×1024 变换 | `(B, 257, 1024)` |
| `x = torch.mul(x, c)` | 广播到 257 个 token，逐通道相乘 | `(B, 257, 1024)` |
| `x[:, 1:, :].unsqueeze(1)` | 丢掉 CLS，只留 patch | `(B, 1, 256, 1024)` |
| `VT_Extractor(...)` | 4 latent × 6 层 | `(B, 1, 4, 1024)` |
| `avgpool((4,1))` | 4 个池成 1 个 | `(B, 1, 1, 1024)` |
| `squeeze ×2` | 去掉多余维度 | `(B, 1024)` |
| `ln_post_local → @proj_local` | 1024 → 768 | `(B, 768)` |
| `layers_local → fc_out_local` | MLP | `(B, 768)` |
| `x_global`（CLS 走另一条路） | 同样 1024 → 768 | `(B, 768)` |
| `x_global + x_local * tanh(gate)` | 加权求和 | `(B, 768)` |

### 两处最容易被误读的地方

**① `torch.mul(x, c)` 不是"把文本拼到图像上"**

`c` 的形状是 `(B, 1, 1024)`，广播到 `(B, 257, 1024)`。它对**每一个 token 用同一组
1024 个系数**做逐通道缩放。所以文本的影响是"调整视觉特征每个通道的重要性"，
而不是"往序列里加一个 token"。真正的 token 级融合发生在下一步——伪词被替换进句子。

**② `W_R` 不混合 token**

`nn.Linear` 作用在最后一维，每个 token 独立变换。257 个 token 之间没有交互。
跨 token 的交互全部交给 `VT_Extractor` 里的注意力去做。分工很清楚：

```text
W_R            → 逐 token 的通道级重构（文本语境在这里注入）
VT_Extractor   → 跨 token 的空间聚合（局部信息在这里提取）
```

到这里，VTE 的输出就是一个 **768 维的伪词向量**，和 CLIP 的词嵌入维度一致——
这就是它能被塞回句子的前提。

## 8. 关键一步：把伪词替换进句子

这是整个方法最精巧的地方。注意它是**替换**，不是插入。

句子模板来自 `src/data.py`：

```python
text_with_blank = 'a photo of <|replace|> , {} and {}'.format(cap2, cap1)
```

下面这段是 `CLIP.encode_text_img_retrieval`。`<|replace|>` 的 token id 是 `vocab_size - 3 = 49405`，在 77 个位置里占 1 个。
`encode_text_img_retrieval` 做的就是：把那个位置的**词嵌入**换成伪词，然后照常过文本 Transformer。

因为只是替换、长度还是 77，所以位置编码、EOT 位置全都不变——
这也解释了为什么后面还用原来的 `collect_ind` 去取 EOT。

In [ ]:
# 出处：model/model.py  第 759-790 行（官方源码逐字截取）
    def encode_text_img_retrieval(self, text, img_tokens, split_ind=4, repeat=True):
        # text.shape = [1, n_ctx]
        # img_tokens.shape = [batch_size, d_model]
        if isinstance(img_tokens, tuple):
            b_size = img_tokens[0].shape[0]
        else:
            b_size = img_tokens.shape[0]
        if repeat:
            text = text.repeat(b_size, 1)
        x = self.token_embedding(text).type(self.dtype)  # [batch_size, n_ctx, d_model]
        collect_ind = text == self.end_id
        collect_ind = collect_ind.nonzero()[:, 1]
        ind_insert = text[0] == split_ind
        if isinstance(img_tokens, tuple):
            indexes = ind_insert.nonzero()
            for i, index in enumerate(indexes):
                img = img_tokens[i].view(b_size, 1, -1)
                x = torch.cat([x[:, :index], img, x[:, index + 1:]], dim=1)
        else:
            img_tokens = img_tokens.view(b_size, 1, -1)
            ind_insert = ind_insert.nonzero()[0]
            x = torch.cat([x[:, :ind_insert], img_tokens, x[:, ind_insert + 1:]], dim=1)
        # x = torch.cat([x, torch.zeros_like(x).cuda()[:, :1, :]], dim=1)
        x = x + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)
        # x.shape = [batch_size, n_ctx, transformer.width]
        # take features from the eot embedding (eot_token is the highest number in each sequence)
        x = x[torch.arange(x.size(0)), collect_ind] @ self.text_projection
        return x

### 三处细节

**① `ind_insert = text[0] == split_ind` 只看第 0 行**

说明官方假设同一批里所有句子模板一样，`<|replace|>` 都在同一个位置。
评估时确实如此（模板固定），所以安全。但如果换模板，这里就会错位。

**② `repeat` 参数**

```python
if repeat:
    text = text.repeat(b_size, 1)
```

- 训练/评估里一次给一条句子、多张图 → `repeat=True`
- 评估循环里句子已经是 `(B, 77)` → `repeat=False`

我们在 `reproduce.py` 里调用时用的就是 `repeat=False`。

**③ 池化位置用 `collect_ind`，不是 `collect_ind + 1`**

对比一下 `encode_text_img`（训练用，插入模式）：

```python
# 插入：序列变长了 1，所以 EOT 往后挪一位
x = x[torch.arange(x.size(0)), collect_ind + 1] @ self.text_projection

# 替换：序列长度不变，EOT 还在原处
x = x[torch.arange(x.size(0)), collect_ind] @ self.text_projection
```

这一个 `+1` 的区别，就是"插入"和"替换"的全部差异。读源码时最容易在这里看漏。

## 9. 训练目标：只训适配器，对比学习

`train()` 里根据精度分了两条路，实际用的是 `get_loss_img2text_replace_token`
（`args.precision == "amp"` 时）。先看它怎么把伪词变成文本特征：

In [ ]:
# 出处：src/trainer.py  第 95-101 行（官方源码逐字截取）
def get_text_features(model, token_features, args):
    text = tokenize("a photo of")
    text = text.cuda(args.gpu, non_blocking=True)
    text = text.view(1, -1)
    text = text.repeat(token_features.size(0), 1)
    text_features = model.encode_text_img(text, token_features)
    return text_features

`"a photo of"` + 插入伪词 → 走 `encode_text_img`（插入分支）。

然后是完整的损失函数：

In [ ]:
# 出处：src/trainer.py  第 235-297 行（官方源码逐字截取）
def get_loss_img2text_replace_token(model, img2text, images, loss_img, loss_txt, args, replaced_texts, memory=None):
    text = replaced_texts.cuda(args.gpu, non_blocking=True)
    # print(text)
    with torch.no_grad():
        image_features = model.encode_image(images)
        image_token_features = model.encode_image_tokens(images)
        # print(image_features.shape, image_token_features.shape) # torch.Size([1, 768]) torch.Size([1, 257, 1024])
        text_cls_embeddings = model.encode_text(text)

    # token_features = img2text(image_features)
    token_features = img2text(text_cls_embeddings, image_token_features)
    text_features = get_text_features(model, token_features, args)
    # text_features = get_text_features_replace(model, token_features, replaced_texts, args) get_text_features

    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    logit_scale = model.logit_scale.exp()
    logit_scale = logit_scale.mean()
    if args.distributed and args.aggregate:
        world_size = dist.get_world_size()
        rank = dist.get_rank()

        # We gather tensors from all gpus to get more negatives to contrast with.
        gathered_image_features = [
            torch.zeros_like(image_features) for _ in range(world_size)
        ]
        gathered_text_features = [
            torch.zeros_like(text_features) for _ in range(world_size)
        ]
        dist.all_gather(gathered_image_features, image_features)
        dist.all_gather(gathered_text_features, text_features)

        all_image_features = torch.cat(
            [image_features]
            + gathered_image_features[:rank]
            + gathered_image_features[rank + 1 :]
        )
        all_text_features = torch.cat(
            [text_features]
            + gathered_text_features[:rank]
            + gathered_text_features[rank + 1 :]
        )

        ground_truth = torch.arange(len(all_image_features)).long()
        if args.gpu is not None:
            ground_truth = ground_truth.cuda(args.gpu, non_blocking=True)

        # this is needed to send gradients back everywhere.
        # Image loss.
        logits_per_image = logit_scale * all_image_features @ all_text_features.t()
        loss_img_val = loss_img(logits_per_image, ground_truth)
        logits_per_text = logits_per_image.t()
        loss_txt_val = loss_txt(logits_per_text, ground_truth)
    else:
        ground_truth = torch.arange(len(image_features)).long()
        if args.gpu is not None:
            ground_truth = ground_truth.cuda(args.gpu, non_blocking=True)
        # Image loss.
        logits_per_image = logit_scale * image_features @ text_features.t()
        loss_img_val = loss_img(logits_per_image, ground_truth)
        logits_per_text = logit_scale * text_features @ image_features.t()
        loss_txt_val = loss_txt(logits_per_text, ground_truth)
    total_loss = (loss_img_val + loss_txt_val) / 2

### 逐段读

```python
with torch.no_grad():
    image_features = model.encode_image(images)
    image_token_features = model.encode_image_tokens(images)
    text_cls_embeddings = model.encode_text(text)
```

**CLIP 全部在 `no_grad` 里。** 视觉特征、patch token、文本 CLS 都是常量，
梯度只流经 `img2text`。配合 `main.py` 里优化器只收 `img2text.named_parameters()`，
可以确认：训练时 CLIP 一个参数都不改。

```python
token_features = img2text(text_cls_embeddings, image_token_features)
text_features = get_text_features(model, token_features, args)
```

- 输入：参考图的 patch token + 整句的文本 CLS
- 输出：伪词
- 再把伪词插进 `"a photo of"`，过文本编码器，得到文本特征

```python
logits_per_image = logit_scale * image_features @ text_features.t()
loss_img_val = loss_img(logits_per_image, ground_truth)
logits_per_text = logit_scale * text_features @ image_features.t()
loss_txt_val = loss_txt(logits_per_text, ground_truth)
total_loss = (loss_img_val + loss_txt_val) / 2
```

这是 CLIP 原版的**对称对比损失**：一个 batch 里第 i 条文本应该和第 i 张图最像，
两个方向的交叉熵各算一次再平均。`logit_scale` 是可学习的温度系数，初值 `log(1/0.07)`。

### 一个必须说清楚的细节

`images` 是 `batch[0]`，在 FashionIQ 的 `return_all` 里对应**参考图**。
而 `text` 是 `batch[2]`，是带修改描述的完整模板。

所以训练目标准确说是：**"给定参考图的视觉 token 和这句修改描述，生成的伪词，
要能让 `a photo of <伪词>` 尽量接近参考图的 CLIP 图像特征。"**

**训练数据的真实来源**：官方 README 写的是 Conceptual Captions（CC3M），
代码里由 `CsvDataset_replace` 读取，训练用的是 CSV 里的 `replaced_captions` 列
（`src/data.py:443-468`）——也就是**已经遮掉物体名、保留原句**的那一版描述，
`<|replace|>` 就插在物体名原来的位置。这是论文的第三个贡献点。

FashionIQ 从头到尾只出现在评估里。这就是"零样本"三个字的实际含义。

## 10. 推理：一次组合检索的数据流

评估循环在 `evaluate_fashion` 里，去掉图库编码和指标计算，组合查询就这十几行：

In [ ]:
# 出处：src/eval_utils.py  第 527-548 行（官方源码逐字截取）
        for batch in tqdm(source_loader):
            ref_images, target_images, target_caption, caption_only, answer_paths, ref_names, captions = batch # answer_path: target image path
            for path in answer_paths:
                all_answer_paths.append(path)
            all_reference_names.extend(ref_names)
            all_captions.extend(captions)
            if args.gpu is not None:
                ref_images = ref_images.cuda(args.gpu, non_blocking=True)
                target_images = target_images.cuda(args.gpu, non_blocking=True)
                target_caption = target_caption.cuda(args.gpu, non_blocking=True)
                caption_only = caption_only.cuda(args.gpu, non_blocking=True)
            image_features = m.encode_image(target_images)

            query_image_features = m.encode_image(ref_images)
            query_image_token_features = model.encode_image_tokens(ref_images)

            text_cls_embeddings = model.encode_text(target_caption)
            id_split = tokenize(["<|replace|>"])[0][1]
            caption_features = m.encode_text(target_caption)                            
            # query_image_tokens = img2text(query_image_features)
            query_image_tokens = img2text(text_cls_embeddings, query_image_token_features)
            composed_feature = m.encode_text_img_retrieval(target_caption, query_image_tokens, split_ind=id_split, repeat=False)

### 每条查询要过 5 次 CLIP 前向

| 调用 | 用途 | 是否必须 |
|---|---|---|
| `encode_image(target_images)` | 官方多算的一次 | **不需要**，结果被图库特征覆盖 |
| `encode_image(ref_images)` | 参考图 CLS → mixture 基线用 | 组合检索不需要 |
| `encode_image_tokens(ref_images)` | 参考图 257 token → VTE 输入 | **必须** |
| `encode_text(target_caption)` | 整句 CLS → VTE 输入 | **必须** |
| `encode_text_img_retrieval(...)` | 伪词替换 + 文本编码 → 组合查询 | **必须** |

第一条我们在复现时直接砍掉了（`docs/baseline_reproduction.md` 第 7 节），
结果不受影响——因为图库特征是另外单独算的，这里算出来的 `image_features` 从没参与排序。

### 组合查询的完整链条

```
参考图 ──encode_image_tokens──> (B,257,1024) ─┐
                                              ├─> img2text ──> 伪词 (B,768)
修改文本 ──encode_text────────> (B,768) ──────┘
                                                    │
                                                    ▼
                                   替换 <|replace|> 的词嵌入
                                                    │
                                                    ▼
                                        文本 Transformer (12 层)
                                                    │
                                                    ▼
                                            取 EOT → (B,768)
                                                    │
                                                    ▼
                                            与图库特征做余弦相似度
```

注意最后一步：**伪词自己不是检索向量**，它要经过整个文本 Transformer 之后，
取出来的 EOT 表征才是。这一点常被误解成"直接用伪词检索"。

## 11. 工程实现：`build_model` 如何凭权重反推架构

官方 `eval_retrieval.py` 的做法是 `load("ViT-L/14")` 先下一份 OpenAI 的 CLIP，
再用检查点整个覆盖——下载的权重对结果零贡献，但没网就直接失败。

`build_model` 换了个思路：**从 state_dict 的形状反推超参数**，不需要下载任何东西。

In [ ]:
# 出处：model/model.py  第 850-889 行（官方源码逐字截取）
def build_model(state_dict: dict):
    vit = "visual.proj" in state_dict

    if vit:
        vision_width = state_dict["visual.conv1.weight"].shape[0]
        vision_layers = len(
            [k for k in state_dict.keys() if k.startswith("visual.") and k.endswith(".attn.in_proj_weight")])
        vision_patch_size = state_dict["visual.conv1.weight"].shape[-1]
        grid_size = round((state_dict["visual.positional_embedding"].shape[0] - 1) ** 0.5)
        image_resolution = vision_patch_size * grid_size
    else:
        counts: list = [len(set(k.split(".")[2] for k in state_dict if k.startswith(f"visual.layer{b}"))) for b in
                        [1, 2, 3, 4]]
        vision_layers = tuple(counts)
        vision_width = state_dict["visual.layer1.0.conv1.weight"].shape[0]
        output_width = round((state_dict["visual.attnpool.positional_embedding"].shape[0] - 1) ** 0.5)
        vision_patch_size = None
        assert output_width ** 2 + 1 == state_dict["visual.attnpool.positional_embedding"].shape[0]
        image_resolution = output_width * 32

    embed_dim = state_dict["text_projection"].shape[1]
    context_length = state_dict["positional_embedding"].shape[0]
    vocab_size = state_dict["token_embedding.weight"].shape[0]
    transformer_width = state_dict["ln_final.weight"].shape[0]
    transformer_heads = transformer_width // 64
    transformer_layers = len(set(k.split(".")[2] for k in state_dict if k.startswith(f"transformer.resblocks")))

    model = CLIP(
        embed_dim,
        image_resolution, vision_layers, vision_width, vision_patch_size,
        context_length, vocab_size, transformer_width, transformer_heads, transformer_layers
    )

    for key in ["input_resolution", "context_length", "vocab_size"]:
        if key in state_dict:
            del state_dict[key]

    convert_weights(model)
    model.load_state_dict(state_dict)
    return model.eval()

### 它是怎么"猜"出来的

| 超参数 | 从哪个张量推出来 |
|---|---|
| 是不是 ViT | `"visual.proj" in state_dict` |
| 视觉宽度 1024 | `visual.conv1.weight.shape[0]` |
| 视觉层数 24 | 统计 `visual.*.attn.in_proj_weight` 的个数 |
| patch 大小 14 | `visual.conv1.weight.shape[-1]` |
| 分辨率 224 | `patch_size × sqrt(位置编码长度 - 1)` |
| 嵌入维度 768 | `text_projection.shape[1]` |
| 上下文长度 77 | `positional_embedding.shape[0]` |
| 词表大小 49408 | `token_embedding.weight.shape[0]` |
| 文本宽度 768 | `ln_final.weight.shape[0]` |
| 注意头数 12 | `transformer_width // 64` |
| 文本层数 12 | 统计 `transformer.resblocks.*` 的组数 |

这就是我们在 `reproduce.py` 里不做任何下载、直接从 `epoch_30.pt` 构造模型的方式。

## 12. 速查表

### 形状

| 张量 | 形状 |
|---|---|
| 输入图像 | `(B, 3, 224, 224)` |
| patch token（含 CLS） | `(B, 257, 1024)` |
| 图像 CLS（降维后） | `(B, 768)` |
| 文本 token | `(B, 77, 768)` |
| 文本 CLS | `(B, 768)` |
| 伪词 | `(B, 768)` |
| 组合查询 | `(B, 768)` |
| 图库特征 | `(N, 768)` |

### 三个常见误解

**① "伪词是拼到句子后面的"**
不是。它**替换** `<|replace|>` 那一个 token 的嵌入，序列长度保持 77。
证据：`encode_text_img_retrieval` 里是 `torch.cat([x[:, :ind], img, x[:, ind+1:]], dim=1)`，
前后两段拼起来正好少一个位置。

**② "把图像特征和文本特征加起来就是组合查询"**
那是 `mixture` 基线（`query_image_features + caption_features`），
在 FashionIQ 上 R@10 只有 15.72，远低于 composed 的 23.60。
两者在官方日志里是分开的两行，不要看混。

**③ "训练更新了整个 CLIP"**
没有。CLIP 全程 `no_grad`，优化器只收 `img2text` 的 68.5 M 参数，
CLIP 的 427.6 M 参数一个没动。

---

## 13. 和本项目下一步的关系

现在这张图应该能对上了：

```text
本 notebook 讲完的部分 ──────────────────────────┐
  Context-I2W 初始检索                          │ 论文给的
  伪词生成（IVS + VTE）                         │
  组合查询表示                                  │
────────────────────────────────────────────────┘
                    ↓
我们自己要做的部分
  属性状态转移评价
  反馈信号构造
  IVS 测试时调整   ← 动的就是这个 W_R
```

最后一行是关键：**第 14 步"IVS 测试时调整"，调整的对象就是本文档第 5 节那个
`self.W_R`。** 它的位置、形状、加载方式现在都已经清楚了。